# <font color="#418FDE" size="6.5" uppercase>**Scaling Distributions**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Apply standard, min-max, max-absolute, robust, and sample normalization transformations. 
- Compare quantile and power transformations for skewed numerical features. 
- Choose scaling strategies that respect sparse and dense data constraints. 


## **1. Core Scaling Methods**

### **1.1. Standard Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_01.jpg?v=1783950239" width="250">



>* Centers features with unit standard deviation
>* Prevents large units from dominating models

>* Prevents large-scale features from dominating models
>* Improves distance, gradient, and PCA methods

>* Outliers can distort standard scaling results
>* Fit scaling only on training data



In [ ]:
#@title Python Code - Standard Scaling

# Demonstrate standard scaling with tiny housing data.
# Compare raw units against standardized feature values.
# Keep output short for beginner friendly learning.

import numpy as np
import pandas as pd

# Create dense numerical features with different units.
data = pd.DataFrame({
    "area_m2": [55, 80, 120, 160, 210],
    "distance_km": [1.2, 3.5, 7.0, 12.0, 20.0],
    "age_years": [2, 8, 15, 30, 45],
})

# Validate the tiny table before scaling.
assert data.shape == (5, 3)
assert np.isfinite(data.to_numpy()).all()

# Compute training means for each feature.
means = data.mean(axis=0)
stds = data.std(axis=0, ddof=0)

# Apply the standard scaling formula manually.
scaled = (data - means) / stds
scaled_summary = scaled.agg(["mean", "std"])

# Use population standard deviation for checking.
scaled_population_stds = scaled.std(axis=0, ddof=0)

# Print compact summaries, not full arrays.
print("Original feature means:")
print(means.round(2).to_string())
print("\nOriginal feature standard deviations:")
print(stds.round(2).to_string())
print("\nScaled feature means:")
print(scaled.mean(axis=0).round(2).to_string())
print("\nScaled population standard deviations:")
print(scaled_population_stds.round(2).to_string())



### **1.2. Min Max Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_02.jpg?v=1783950237" width="250">



>* Maps values to a fixed range
>* Makes differently scaled features comparable

>* Preserves distribution shape within fixed bounds
>* Makes varied features comparable and interpretable

>* Extreme values can distort min max scaling
>* Fit scaling only on training data



In [ ]:
#@title Python Code - Min Max Scaling

# Demonstrate min max scaling manually.
# Compare original and scaled feature ranges.
# Keep output short for beginners.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create tiny housing data with different units.
data = pd.DataFrame({
    "size_sqft": [650, 800, 950, 1200, 1500],
    "rent_dollars": [1200, 1500, 1800, 2400, 3000]
})

# Validate the table before scaling.
assert data.shape == (5, 2)
assert data.notna().all().all()

# Compute feature minimums and maximums.
feature_min = data.min(axis=0)
feature_max = data.max(axis=0)
feature_range = feature_max - feature_min

# Prevent division by zero for constant columns.
assert (feature_range > 0).all()
scaled = (data - feature_min) / feature_range

# Combine rounded values for compact viewing.
summary = pd.concat(
    [data, scaled.round(3).add_suffix("_scaled")],
    axis=1
)

# Print only a compact teaching summary.
print("Original minimums:", feature_min.to_dict())
print("Original maximums:", feature_max.to_dict())
print("Scaled minimums:", scaled.min().round(3).to_dict())
print("Scaled maximums:", scaled.max().round(3).to_dict())
print(summary.head().to_string(index=False))

# Plot original rent against scaled rent.
plt.figure(figsize=(6, 4))
plt.plot(data["rent_dollars"], scaled["rent_dollars"], marker="o")
plt.xlabel("Original rent in dollars")
plt.ylabel("Min max scaled rent")
plt.title("Min Max Scaling Preserves Order")
plt.grid(True, alpha=0.3)
plt.show()



### **1.3. Max Absolute Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_03.jpg?v=1783950241" width="250">



>* Scales values by largest absolute feature value
>* Preserves signs without centering or shifting

>* Keeps zeros unchanged in sparse datasets
>* Supports efficient high-dimensional data processing

>* Outliers can compress most scaled values
>* Inspect data; use robust scaling if needed



In [ ]:
#@title Python Code - Max Absolute Scaling

# Demonstrate max absolute scaling clearly.
# Preserve signs and zero values.
# Compare original and scaled features.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create small profit and loss data.
data = pd.DataFrame(
    {
        "store_profit_dollars": [-5000, -1200, 0, 1800, 4200],
        "ad_clicks_sparse": [0, 3, 0, 12, 0],
    }
)

# Validate the tiny table shape.
assert data.shape == (5, 2)

# Find each feature's largest absolute value.
max_abs_values = data.abs().max(axis=0)

# Avoid division by zero safely.
safe_denominators = max_abs_values.replace(0, 1)

# Scale values by maximum absolute magnitude.
scaled_data = data.divide(safe_denominators, axis=1)

# Count zeros before and after scaling.
zeros_before = int((data == 0).sum().sum())
zeros_after = int((scaled_data == 0).sum().sum())

# Print a compact comparison table.
print("Original data:")
print(data.to_string(index=False))
print("\nMax absolute scaled data:")
print(scaled_data.round(3).to_string(index=False))
print(f"\nZeros before: {zeros_before}; zeros after: {zeros_after}")
print("Signs stay negative, zero, or positive after scaling.")

# Plot original and scaled profit values.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))

# Show original profit magnitudes.
axes[0].bar(data.index, data["store_profit_dollars"], color="steelblue")
axes[0].set_title("Original profit dollars")
axes[0].axhline(0, color="black", linewidth=1)

# Show scaled profit magnitudes.
axes[1].bar(scaled_data.index, scaled_data["store_profit_dollars"], color="orange")
axes[1].set_title("Max absolute scaled")
axes[1].axhline(0, color="black", linewidth=1)

# Keep the plot readable.
plt.tight_layout()
plt.show()



## **2. Robust Transforming**

### **2.1. Robust Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_01.jpg?v=1783950219" width="250">



>* Scale using medians and quartile ranges
>* Limit outlier influence in skewed features

>* Robust scaling preserves distribution shape
>* Quantile and power transformations reshape skew

>* Middle values define the typical scale
>* Use other transforms to fix skew



In [ ]:
#@title Python Code - Robust Scaling

# Robust scaling reduces outlier influence.
# Median and quartiles define typical spread.
# This example uses salary values.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create skewed salaries with extreme executives.
salaries = np.array([42, 45, 47, 50, 52, 55, 58, 62, 68, 75, 220, 480], dtype=float)

# Validate the tiny teaching dataset.
assert salaries.size >= 4 and np.isfinite(salaries).all()

# Compute standard scaling using mean spread.
standard_scaled = (salaries - salaries.mean()) / salaries.std()

# Compute robust scaling using middle spread.
median_value = np.median(salaries)
quartile_one, quartile_three = np.percentile(salaries, [25, 75])

# Protect against zero interquartile range.
iqr_value = max(quartile_three - quartile_one, 1e-12)
robust_scaled = (salaries - median_value) / iqr_value

# Summarize only key values for learners.
summary = pd.DataFrame({"salary_k": salaries, "standard": standard_scaled, "robust": robust_scaled})
print("Median salary and IQR define robust scaling.")
print(f"Median: {median_value:.1f}k, IQR: {iqr_value:.1f}k")
print(summary.round(2).head(4).to_string(index=False))
print(summary.round(2).tail(2).to_string(index=False))

# Plot original and scaled distributions together.
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].boxplot(salaries, vert=False)
axes[1].boxplot(standard_scaled, vert=False)
axes[2].boxplot(robust_scaled, vert=False)

# Add short titles for comparison.
axes[0].set_title("Original salaries")
axes[1].set_title("Standard scaled")
axes[2].set_title("Robust scaled")

# Label the shared measurement idea.
for axis in axes:
    axis.set_xlabel("Transformed value")

plt.tight_layout()
plt.show()



### **2.2. Sample Normalization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_02.jpg?v=1783950225" width="250">



>* Rescales each row by overall magnitude
>* Highlights proportions over absolute sample size

>* Quantile and power reduce feature skewness
>* Sample normalization compares within-row proportions

>* Normalize profiles when relative patterns matter
>* Avoid discarding meaningful total magnitude



In [ ]:
#@title Python Code - Sample Normalization

# Sample normalization rescales each row.
# It emphasizes composition over magnitude.
# This example uses tiny purchase profiles.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create small customer purchase counts.
customers = ["Low total", "High total", "Different mix"]
features = ["Books", "Games", "Music"]
raw_values = np.array([[2, 3, 5], [20, 30, 50], [8, 1, 1]], dtype=float)

# Validate the tiny dense matrix.
assert raw_values.shape == (3, 3)
assert np.all(raw_values.sum(axis=1) > 0)

# Normalize each sample by its row sum.
row_totals = raw_values.sum(axis=1, keepdims=True)
normalized_values = raw_values / row_totals

# Build compact tables for display.
raw_table = pd.DataFrame(raw_values, index=customers, columns=features)
norm_table = pd.DataFrame(normalized_values, index=customers, columns=features)

# Print only concise teaching output.
print("Raw row totals:", raw_table.sum(axis=1).astype(int).to_dict())
print("Normalized row totals:", norm_table.sum(axis=1).round(1).to_dict())
print("Low and high totals now match:", np.allclose(norm_table.iloc[0], norm_table.iloc[1]))
print("Different mix stays different:", norm_table.iloc[2].round(2).to_dict())

# Plot normalized within-sample composition.
ax = norm_table.plot(kind="bar", stacked=True, figsize=(7, 4))
ax.set_title("Sample normalization compares within-row composition")
ax.set_ylabel("Share of each customer's purchases")
ax.set_xlabel("Customer profile")

# Keep the single plot readable.
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()



### **2.3. Sparse Safe Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_03.jpg?v=1783950228" width="250">



>* Keep sparse zeros unchanged during scaling
>* Avoid centering that destroys sparse efficiency

>* Quantile transforms may disrupt sparse zeros
>* Power transforms can preserve sparse structure

>* Balance statistical goals with sparse storage limits
>* Keep zeros unchanged; avoid centering transformations



In [ ]:
#@title Python Code - Sparse Safe Scaling

# Sparse scaling protects meaningful zero entries.
# Centering sparse data can destroy sparsity.
# This example compares safe transformations.

import numpy as np

# Create tiny sparse-like count data.
counts = np.array([
    [0, 0, 3, 0],
    [0, 5, 0, 0],
    [2, 0, 0, 9],
    [0, 0, 0, 1],
], dtype=float)

# Count zeros before any transformation.
zero_count = int(np.sum(counts == 0))
total_count = int(counts.size)

# Mean centering changes stored zeros.
centered = counts - counts.mean(axis=0)
centered_zeros = int(np.sum(centered == 0))

# Max absolute scaling preserves zeros.
max_abs = np.max(np.abs(counts), axis=0)
safe_denominator = np.where(max_abs == 0, 1, max_abs)
maxabs_scaled = counts / safe_denominator

# Log compression preserves absent words.
log_scaled = np.where(counts > 0, np.log1p(counts), 0)
log_zeros = int(np.sum(log_scaled == 0))

# Summarize sparse safety clearly.
print(f"Original zeros: {zero_count} of {total_count}")
print(f"After mean centering zeros: {centered_zeros} of {total_count}")
print(f"After max-absolute scaling zeros: {int(np.sum(maxabs_scaled == 0))} of {total_count}")
print(f"After log compression zeros: {log_zeros} of {total_count}")
print("Lesson: avoid centering when zeros mean absence.")



## **3. Distribution Shape**

### **3.1. Quantile Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_01.jpg?v=1783950235" width="250">



>* Scale values by rank, not distance
>* Handle skewed features and extreme outliers

>* Choose uniform or normal target distributions
>* Rank reshaping changes distances and interpretability

>* Use quantile scaling mainly for dense features
>* Preserve sparse zeros to avoid costly densification



In [ ]:
#@title Python Code - Quantile Scaling

# Demonstrate quantile scaling with ranks.
# Compare dense and sparse feature behavior.
# Keep output short for beginners.

import numpy as np
import matplotlib.pyplot as plt

# Create a skewed dense spending feature.
spending = np.array([8, 10, 12, 15, 18, 25, 40, 90, 220, 600])

# Create sparse click counts with many zeros.
clicks = np.array([0, 0, 0, 0, 0, 1, 0, 2, 0, 5])

# Define a simple rank based scaler.
def quantile_uniform(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values, kind="mergesort")

    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(values), dtype=float)
    scaled = ranks / max(len(values) - 1, 1)

    return scaled

# Transform both example features.
spending_q = quantile_uniform(spending)
clicks_q = quantile_uniform(clicks)

# Count zeros before and after scaling.
clicks_zero_before = int(np.sum(clicks == 0))
clicks_zero_after = int(np.sum(clicks_q == 0))

# Print a compact teaching summary.
print("Dense spending values spread evenly by rank.")
print("Original spending:", spending.tolist())
print("Quantile scaled:", np.round(spending_q, 2).tolist())
print("Sparse zeros before scaling:", clicks_zero_before)
print("Sparse zeros after scaling:", clicks_zero_after)
print("Lesson: quantile scaling can destroy sparsity.")

# Plot original versus quantile scaled spending.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(spending, bins=6, color="steelblue")
axes[0].set_title("Original skewed spending")
axes[1].hist(spending_q, bins=6, color="darkorange")

axes[1].set_title("Quantile scaled ranks")
for axis in axes:
    axis.set_ylabel("Count")
    axis.set_xlabel("Value")

plt.tight_layout()
plt.show()



### **3.2. Power Transform**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_02.jpg?v=1783950230" width="250">



>* Makes skewed distributions more balanced
>* Compresses large values for better modeling

>* Dense features usually suit power transforms
>* Sparse features need zero-preserving caution

>* Respect zeros and negative values
>* Prefer dense features; preserve sparse zeros



In [ ]:
#@title Python Code - Power Transform

# Power transforms reshape skewed dense measurements.
# Sparse zeros need careful preprocessing choices.
# This example avoids scikit-learn intentionally.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic skewed positive dense data.
rng = np.random.default_rng(42)
values = rng.lognormal(mean=2.0, sigma=0.9, size=120)

# Apply a simple Box-Cox style transform.
lambda_value = 0.25
powered = (np.power(values, lambda_value) - 1) / lambda_value

# Compare skewness before and after transformation.
def skewness(x):
    centered = x - np.mean(x)
    return np.mean(centered**3) / np.std(x)**3

# Demonstrate sparse zero preservation concern.
sparse_like = np.array([0, 0, 0, 2, 8, 30], dtype=float)
shifted_power = np.power(sparse_like + 1, lambda_value)

# Print only compact teaching summaries.
print(f"Original dense skewness: {skewness(values):.2f}")
print(f"Power transformed skewness: {skewness(powered):.2f}")
print(f"Sparse-like zeros before: {np.sum(sparse_like == 0)}")
print(f"Zeros after shifted power: {np.sum(shifted_power == 0)}")
print("Dense skewed measurements often benefit most.")
print("Sparse indicators may lose efficient zero structure.")

# Plot compact histograms for visual comparison.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(values, bins=18, color="steelblue", edgecolor="white")
axes[0].set_title("Original skewed dense feature")
axes[1].hist(powered, bins=18, color="seagreen", edgecolor="white")
axes[1].set_title("After power transform")
plt.tight_layout()
plt.show()



### **3.3. Distribution Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_03.jpg?v=1783950232" width="250">



>* Quantile scaling reshapes features using value ranks
>* Power transforms reduce skew while preserving spacing

>* Dense features can use distribution transformations directly
>* Sparse zeros need meaning and memory preserved

>* Match transformations to feature shape
>* Preserve sparse zeros and useful structure



In [ ]:
#@title Python Code - Distribution Comparison

# Compare dense and sparse distribution choices.
# Quantile reshapes ranks, power smooths skew.
# Sparse zeros need special protection.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create deterministic skewed dense spending values.
rng = np.random.default_rng(7)
spending = rng.lognormal(mean=3.0, sigma=1.0, size=80)

# Add repeated zeros to mimic sparse counts.
counts = np.zeros(80, dtype=float)
counts[rng.choice(80, 12, replace=False)] = rng.integers(1, 9, 12)

# Define simple rank based quantile transform.
def quantile_normal_like(values):
    ranks = pd.Series(values).rank(method="average").to_numpy()
    scaled = (ranks - 0.5) / len(values)
    return np.log(scaled / (1.0 - scaled))

# Define safe log power style transform.
def log_power_like(values):
    return np.log1p(np.maximum(values, 0.0))

# Transform dense and sparse examples.
dense_quantile = quantile_normal_like(spending)
dense_power = log_power_like(spending)
sparse_quantile = quantile_normal_like(counts)
sparse_power = log_power_like(counts)

# Count how many zeros remain exactly zero.
original_zeros = int(np.sum(counts == 0.0))
quantile_zeros = int(np.sum(sparse_quantile == 0.0))
power_zeros = int(np.sum(sparse_power == 0.0))

# Print a compact comparison summary.
print("Dense spending: compare shape changes in the plot.")
print(f"Sparse original zeros: {original_zeros} of {counts.size}.")
print(f"After quantile transform, exact zeros: {quantile_zeros}.")
print(f"After log power transform, exact zeros: {power_zeros}.")
print("Lesson: dense skew can be reshaped; sparse zeros need protection.")

# Plot original and transformed dense distributions.
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].hist(spending, bins=15, color="steelblue")
axes[0].set_title("Original dense")

# Show quantile transformed dense distribution.
axes[1].hist(dense_quantile, bins=15, color="darkorange")
axes[1].set_title("Quantile shaped")
axes[2].hist(dense_power, bins=15, color="seagreen")
axes[2].set_title("Power shaped")

# Keep the plot readable in Colab.
for axis in axes:
    axis.set_yticks([])
    axis.set_xlabel("value")

plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Scaling Distributions**</font>


In this lecture, you learned to:
- Apply standard, min-max, max-absolute, robust, and sample normalization transformations. 
- Compare quantile and power transformations for skewed numerical features. 
- Choose scaling strategies that respect sparse and dense data constraints. 

In the next Lecture (Lecture B), we will go over 'Expansion Binning'